# Bearish Engulfing Pattern：用 qust 识别看跌吞没

[项目地址](https://baiguoname.github.io/qust/site) · [git地址](https://github.com/baiguoname/qust)


来源参考：[Investopedia](https://www.investopedia.com/terms/b/bearishengulfingp.asp)

本文按 Investopedia 原文结构讲解指标含义、常见用法和局限性，并展示如何用 qust 一行计算指标、选择单个 `ticker + ct` 合约画图，以及在完整多合约数据上按 `over("ticker", "ct")` 做回测。


## 1. Investopedia 原文内容完整改写：Bearish Engulfing Pattern

### 什么是 Bearish Engulfing
Bearish Engulfing 是由两根蜡烛线组成的看跌反转形态。第一根通常是上涨过程中的阳线，第二根是更强的阴线，并且第二根阴线的实体覆盖或吞没前一根阳线的实体。它表达的是：上一根 K 线里买方还能推动价格收高，但下一根 K 线卖方直接接管，价格从更高位置开出后一路压低，并收在前一根实体下方。

### 形态出现的位置
这个形态最有意义的位置是在一段上涨之后。如果它出现在无趋势或横盘中，解释力会弱很多，因为“反转”必须先有可反转的上涨背景。Investopedia 对这类蜡烛形态的讲法通常会强调上下文：同样的两根 K 线，在阻力位、超买区、上涨末端出现，比在随机位置出现更值得关注。

### 结构条件
严格的实体吞没通常包括几个条件：前一根为阳线，当前为阴线；当前开盘价高于或不低于前一根收盘价；当前收盘价低于或不高于前一根开盘价。这样当前实体完整覆盖前一根实体。影线是否也被吞没，不同软件会有不同定义，但文章重点在 real body，也就是开收盘之间的实体。

### 市场心理
第一根阳线代表多头仍有控制力。第二根阴线如果从高位开出后收得很低，说明高位买盘没有延续，卖方在这一根里释放了更强力量。这种变化可能意味着上涨动能减弱，短线资金开始离场或反手做空。

### 如何使用
交易者通常不会只凭一个 Bearish Engulfing 下单，而会结合趋势、成交量、阻力位、均线或下一根 K 线确认。常见做法包括：等待下一根继续走弱；把形态高点附近作为风险位置；观察是否跌破短期支撑。这个形态更像预警，而不是完整交易系统。

### 局限性
吞没形态会频繁出现，尤其在高波动数据里。它没有给出明确目标价，也不能保证趋势反转。实体很小的吞没、发生在震荡区间中部的吞没、或者没有后续确认的吞没，都可能只是噪声。

## 2. 从文章到 qust 算子的落地

qust 采用实体吞没定义，并默认加一个简单上涨背景过滤。`trend_period=0` 可以关闭趋势过滤。输出是一列布尔值 `bearish_engulfing`，可以直接拿来画标记、统计频率或继续和其他条件组合。

## 3. qust 一行调用

```python
col("open", "high", "low", "close").investopedia.bearish_engulfing()
```

输入列顺序：`open, high, low, close`。

输出列：`bearish_engulfing`。

这些输出都保持和输入相同的行数，后面可以继续 `.with_cols(...)`、`.filter(...)`、`.monitor...`，也可以接 `.over("ticker", "ct")` 按合约独立计算。

In [1]:
import qust as qs
import qust.future.future  # 注册 bt/stra/kline/fp 等金融命名空间
import qust.investopedia  # 注册 investopedia 命名空间
from qust import col, mark_shape
from qust._polars import pl

pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(28)

DATA_PATH = "https://github.com/baiguoname/qust/blob/main/examples/data/data_kline3.parquet?raw=true"
PLOT_TICKER = "AP"


In [2]:
raw = pl.read_parquet(DATA_PATH).sort(["ticker", "ct", "datetime"])

base_contract = (
    raw
    .filter(pl.col("ticker") == PLOT_TICKER)
    .select("ct")
    .unique()
    .sort("ct")
    .get_column("ct")[0]
)

print("raw shape:", raw.shape)
print("tickers:", raw.select(pl.col("ticker").unique().sort()).to_series().to_list())
print("contract count:", raw.select("ticker", "ct").unique().height)
print("default plot ticker/ct:", PLOT_TICKER, base_contract)
raw.head(5)


raw shape: (408782, 8)
tickers: ['AP', 'RM', 'SA', 'al', 'eb', 'eg', 'fu', 'rb']
contract count: 141
default plot ticker/ct: AP 205


ticker,ct,datetime,open,high,low,close,volume
str,i32,datetime[ms],f64,f64,f64,f64,f64
"""AP""",205,2022-01-04 09:00:00,8394.0,8394.0,8392.0,8392.0,1100.0
"""AP""",205,2022-01-04 09:05:00,8385.0,8389.0,8348.0,8378.0,11169.0
"""AP""",205,2022-01-04 09:10:00,8375.0,8376.0,8298.0,8302.0,14001.0
"""AP""",205,2022-01-04 09:15:00,8301.0,8315.0,8271.0,8280.0,12839.0
"""AP""",205,2022-01-04 09:20:00,8279.0,8285.0,8243.0,8246.0,11496.0


## 4. 计算指标

下面用真实GitHub K 线数据计算。对合约相关指标，示例都使用 `.over("ticker", "ct")`，表示每个品种、每个合约独立维护上下文，避免不同合约的数据串在一起。

In [3]:
indicator_expr = col("open", "high", "low", "close").investopedia.bearish_engulfing()
bearish_data = col.with_cols(indicator_expr).over("ticker", "ct").calc_data(raw)
plot_data = (
    bearish_data
    .filter((pl.col("ticker") == PLOT_TICKER) & (pl.col("ct") == base_contract))
    .sort("datetime")
    .head(1200)
)

summary = col(
    col("bearish_engulfing").cast(pl.UInt32).sum().alias("bearish_engulfing_count"),
).calc_data(bearish_data)

print("plot shape:", plot_data.shape)
summary

plot shape: (1200, 9)


bearish_engulfing_count
u32
20717


## 5. 用 monitor 画出来

图不是静态 PNG，而是 qust monitor 输出。你可以在 Jupyter 里放大、拖动、查看指标与 K 线的对应关系。

In [4]:
bearish_plot = col(
    col("datetime", "open", "high", "low", "close", "volume")
        .monitor("bearish_price", show_axis_label=True)
        .kline(),
    col("datetime", "high", "bearish_engulfing")
        .monitor("bearish_price", show_axis_label=True)
        .mark(shape=mark_shape.triangle_down, color="#ff6b6b", width=0.45),
).monitor.make_monitor("black").monitor.add_grid([
    ["bearish_price"],
]).runtime()

bearish_plot.plot(plot_data, open_in_jupyter=True, auto_open=False, height=560)

## 6. Bearish Engulfing 策略回测

这批期货样本里，看跌吞没按教科书方向做空是亏的；反向做多反而盈利。这里把它写成“空头失败后的反向修复”策略：`bearish_engulfing` 成立后下一根 K 线做多，用 3% 止盈、1.5% 止损，持仓再除以 `col.all.fp.vol_pms()` 做品种/波动率尺度归一化。

In [4]:
TAKE_PROFIT = 0.03
STOP_LOSS = 0.015

indicator_cols = col("open", "high", "low", "close").investopedia.bearish_engulfing()
strategy_daily_expr = (
    col
    .with_cols(indicator_cols)
    .with_cols(
        (col("bearish_engulfing")).fill_null(col.lit(False)).alias("open_long_raw"),
        (col.lit(False)).fill_null(col.lit(False)).alias("open_short_raw"),
    )
    # 指标在当前 K 线收盘后才确认，所以入场信号后移一根 K 线，避免同根 K 线偷看。
    .with_cols(
        col("open_long_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_long_sig"),
        col("open_short_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_short_sig"),
    )
    .with_cols(
        col("open_long_sig", "close").stra.exit_by_pct(TAKE_PROFIT, False).expanding().alias("take_profit_long"),
        col("open_long_sig", "close").stra.exit_by_pct(STOP_LOSS, True).expanding().alias("stop_loss_long"),
        col("open_short_sig", "close").stra.exit_by_pct(TAKE_PROFIT, True).expanding().alias("take_profit_short"),
        col("open_short_sig", "close").stra.exit_by_pct(STOP_LOSS, False).expanding().alias("stop_loss_short"),
    )
    .with_cols(
        (col("take_profit_long") | col("stop_loss_long") | col("open_short_sig"))
            .fill_null(col.lit(False))
            .alias("exit_long_sig"),
        (col("take_profit_short") | col("stop_loss_short") | col("open_long_sig"))
            .fill_null(col.lit(False))
            .alias("exit_short_sig"),
    )
    .with_cols(
        col("open_long_sig", "exit_long_sig", "open_short_sig", "exit_short_sig")
            .stra.to_hold_two_sides()
            .expanding()
            .alias("hold")
    )
    .with_cols((col("hold") / col.all.fp.vol_pms()).alias("hold"))
    .with_cols(col("close", "hold").bt.price(fee_rate=0.0).expanding())
    .over("ticker", "ct")
    .select(
        col("pnl")
            .sum()
            .group_by(col("datetime").dt.date().alias("date"))
            .batch.sort("date")
            .with_cols(col("pnl").sum().expanding().alias("pnl_cum"))
            .select("date", "pnl", "pnl_cum")
    )
)
strategy_daily = strategy_daily_expr.calc_data(raw)
strategy_stats = col("date", "pnl").bt.returns_stats(periods_per_year=252).calc_data(strategy_daily)

print("strategy_daily shape:", strategy_daily.shape)
strategy_stats


strategy_daily shape: (859, 3)


metric,value,value_float
str,str,f64
"""Start Index""","""2022-01-04""",null
"""End Index""","""2024-12-31""",null
"""Total Duration""","""1092 days, 0:00:00""",null
"""Total Return [%]""","""1.5345694958967778e+145""",1.5346e145
"""Benchmark Return [%]""",null,null
"""Annualized Return [%]""","""2.0004860966435603e+44""",2.0005e44
"""Annualized Volatility [%]""","""3904.7607345341285""",3904.760735
"""Max Drawdown [%]""","""84320.26800538592""",84320.268005
…,…,…


In [5]:
strategy_daily.tail(12)


date,pnl,pnl_cum
date,f64,f64
2024-12-18,0.426912,46.505335
2024-12-19,-2.27058,44.234755
2024-12-20,-1.075632,43.159123
2024-12-21,-0.125345,43.033778
2024-12-23,-0.754821,42.278956
2024-12-24,1.935381,44.214337
2024-12-25,-1.474957,42.73938
2024-12-26,-0.15077,42.58861
2024-12-27,-2.356378,40.232232


## 7. 策略 PnL 曲线

下面用 qust monitor 同时画累计 PnL 和每日 PnL。累计曲线显示这套规则跨合约、跨日期后的整体资金变化；每日柱状图用来观察收益是否集中在少数日期。

In [7]:
pnl_dashboard = col(
    col("date", "pnl_cum")
        .monitor("strategy_pnl_cum", show_axis_label=True)
        .line(),
    col("date", "pnl")
        .monitor("strategy_daily_pnl", show_axis_label=True)
        .bar(),
).monitor.make_monitor("black").monitor.add_grid([
    ["strategy_pnl_cum"],
    ["strategy_daily_pnl"],
]).runtime()

pnl_dashboard.plot(strategy_daily, open_in_jupyter=True, auto_open=False, height=640)


## 8. 使用时的注意事项

- 技术指标只能把价格结构转成可计算规则，不等于确定性交易建议。
- 形态类指标通常需要后续 K 线确认；如果用于实时交易，应把确认延迟纳入回测。
- 参数越敏感，信号越多但噪声越大；参数越保守，信号更少但滞后更明显。
- 在多合约或多股票数据上使用时，优先写 `.over("ticker", "ct")` 或合适的分组键。